In [ ]:
import getpass
token = getpass.getpass("GitHub token: ")
!git clone --branch 43-train-the-distilroberta-multi-label-classifier https://{token}@github.com/Break-Through-Tech/Accenture-1Q-contract-review-challenge.git repo
%cd repo
!ls


GitHub token: ··········
Cloning into 'repo'...
remote: Enumerating objects: 300, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 300 (delta 71), reused 78 (delta 38), pack-reused 175 (from 1)
Receiving objects: 100% (300/300), 31.70 MiB | 15.50 MiB/s, done.
Resolving deltas: 100% (132/132), done.
/content/repo
app			       docs			       reports
Challenge-Project-Overview.md  Getting-Started-for-Fellows.md  requirements.txt
configs			       notebooks		       results
data			       README.md		       src


In [ ]:
!pwd
!ls -la
!git log --oneline -1


/content/repo
total 72
drwxr-xr-x 11 root root  4096 Nov  3 23:51 .
drwxr-xr-x  1 root root  4096 Nov  3 23:51 ..
drwxr-xr-x  2 root root  4096 Nov  3 23:51 app
-rw-r--r--  1 root root 10047 Nov  3 23:51 Challenge-Project-Overview.md
drwxr-xr-x  2 root root  4096 Nov  3 23:51 configs
drwxr-xr-x  4 root root  4096 Nov  3 23:51 data
drwxr-xr-x  2 root root  4096 Nov  3 23:51 docs
-rw-r--r--  1 root root  1798 Nov  3 23:51 Getting-Started-for-Fellows.md
drwxr-xr-x  8 root root  4096 Nov  3 23:51 .git
-rw-r--r--  1 root root   504 Nov  3 23:51 .gitignore
drwxr-xr-x  2 root root  4096 Nov  3 23:51 notebooks
-rw-r--r--  1 root root  4027 Nov  3 23:51 README.md
drwxr-xr-x  3 root root  4096 Nov  3 23:51 reports
-rw-r--r--  1 root root   172 Nov  3 23:51 requirements.txt
drwxr-xr-x  2 root root  4096 Nov  3 23:51 results
drwxr-xr-x  8 root root  4096 Nov  3 23:51 src
d40a672 (HEAD -> 43-train-the-distilroberta-multi-label-classifier, origin/43-train-the-distilroberta-multi-label-classifier) Bu

In [3]:
!pip install -q transformers pyyaml
import torch; print("cuda:", torch.cuda.is_available())



cuda: True


In [4]:
!python -m src.ingest.build_dataset


config.json: 100% 480/480 [00:00<00:00, 2.26MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 63.2kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 10.2MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 3.92MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 15.1MB/s]
chunks: 17,324
split
test      2938
train    14386
max tokens in any chunk (cleaned): 510

categories: 18,054 contract/category rows, 41 distinct

label columns:        41
positive in train:    34   <- trainer's num_labels
no positive in train: ['Affiliate License-Licensor', 'Most Favored Nation', 'Non-Disparagement', 'Price Restrictions', 'Source Code Escrow', 'Third Party Beneficiary', 'Unlimited/All-You-Can-Eat-License']

contract/category pairs by status:
status
absent    11492
placed     6562

spans by placement:
status
placed    13544

unknown labels: 100,702 of 710,284 (14.2%)

wrote chunks.parquet, chunk_labels.parquet and label_audit/ to /content/repo/data/processed


In [5]:
from src.models.trainer import train_and_evaluate
result = train_and_evaluate("configs/transformer.yaml", device="cuda")


model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


epoch 1/3 - train loss: 0.0887
epoch 2/3 - train loss: 0.0470
epoch 3/3 - train loss: 0.0353


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

test summary: {
  "threshold": 0.5,
  "n_categories": 34,
  "macro_precision": 0.8110330165120745,
  "macro_recall": 0.6994659154983746,
  "macro_f1": 0.717027716485196,
  "micro_precision": 0.8691189050470488,
  "micro_recall": 0.8341543513957307,
  "micro_f1": 0.851277754503561
}


In [6]:
import pandas as pd
pd.set_option("display.max_rows", 50)
metrics = result["per_category_metrics"].sort_values("f1", ascending=False)
print(result["summary"])
metrics


{'threshold': 0.5, 'n_categories': 34, 'macro_precision': 0.8110330165120745, 'macro_recall': 0.6994659154983746, 'macro_f1': 0.717027716485196, 'micro_precision': 0.8691189050470488, 'micro_recall': 0.8341543513957307, 'micro_f1': 0.851277754503561}


,category,precision,recall,f1,support,predicted_positive
25,Parties,1.000000,0.980392,0.990099,102,100
8,Document Name,1.000000,0.950980,0.974874,102,97
13,Insurance,0.941176,1.000000,0.969697,32,34
12,Governing Law,0.987500,0.951807,0.969325,83,80
2,Anti-Assignment,1.000000,0.916667,0.956522,72,66
11,Expiration Date,0.915663,0.974359,0.944099,78,83
17,License Grant,0.958333,0.920000,0.938776,50,48
1,Agreement Date,0.915789,0.935484,0.925532,93,95
3,Audit Rights,0.916667,0.868421,0.891892,38,36
4,Cap On Liability,0.854167,0.931818,0.891304,44,48


In [7]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/cuad-runs"
!cp -r models/distilroberta-cuad "/content/drive/MyDrive/cuad-runs/"


Mounted at /content/drive


In [8]:
from google.colab import files
files.download("models/distilroberta-cuad/test_category_metrics.csv")
files.download("models/distilroberta-cuad/run_metadata.json")
files.download("models/distilroberta-cuad/training_history.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>